# Multi Domain - OFDM System Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky

# --- System Parameters ---
num_subcarriers = 64
num_ofdm_symbols_ber = 5000 # Renamed for clarity for BER simulation
num_ofdm_symbols_papr = 10000 # Number of OFDM symbols for PAPR statistics
mod_order = 4  # QPSK
bits_per_symbol = int(np.log2(mod_order))
snr_range_db = np.arange(0, 26, 2)

# --- Plotting Font Size ---
plot_font_size = 16 # Increased font size for all plot text
legend_font_size = 14
title_font_size = 16

# --- Power Parameters (mW) ---
P_RF = 100    # Power per RF chain
P_MMSE = 0.1  # MMSE processing power coefficient (per subcarrier per Nt^3)
P_SEL = 0.01  # Antenna selection power coefficient (per subcarrier per Nt)

# --- Channel Parameters ---
tx_corr = 0.3  # Tx antenna correlation (0-1)
rx_corr = 0.1  # Rx antenna correlation

# --- QPSK Modulation/Demodulation ---
def qpsk_modulate(bits, scale=1.0):
    """QPSK modulation function."""
    if len(bits) % 2 != 0:
        raise ValueError("Number of bits for QPSK modulation must be even.")
    bits = bits.reshape(-1, 2)
    symbols = (2*bits[:,0]-1 + 1j*(2*bits[:,1]-1)) / np.sqrt(2) * scale
    return symbols

def qpsk_demodulate(symbols):
    """QPSK demodulation function."""
    bits = np.zeros(2*len(symbols), dtype=int)
    bits[::2] = (np.real(symbols) > 0).astype(int)
    bits[1::2] = (np.imag(symbols) > 0).astype(int)
    return bits

# --- Generate Correlated Channel ---
def generate_correlated_channel(num_rx, num_tx, num_sc, tx_corr, rx_corr):
    """Generates a correlated MIMO fading channel matrix."""
    # Kronecker model for correlation
    R_tx = (1 - tx_corr)*np.eye(num_tx) + tx_corr*np.ones((num_tx,num_tx))
    R_rx = (1 - rx_corr)*np.eye(num_rx) + rx_corr*np.ones((num_rx,num_rx))
    
    H = (np.random.randn(num_rx, num_tx, num_sc) + 
         1j*np.random.randn(num_rx, num_tx, num_sc)) / np.sqrt(2) # Initial uncorrelated Rayleigh
    
    # Apply correlation using Cholesky decomposition
    L_tx = cholesky(R_tx, lower=True)
    L_rx = cholesky(R_rx, lower=True)
    for k in range(num_sc):
        H[:,:,k] = L_rx @ H[:,:,k] @ L_tx.conj().T # Use conj().T for Hermitian conjugate
        
    # Normalize for unit average gain (average power of H elements is 1)
    return H / np.sqrt(num_tx + num_rx)


# --- MMSE MIMO-OFDM Simulation ---
def simulate_mmse_mimo(num_tx, num_rx, num_syms):
    """Simulates MMSE MIMO-OFDM for BER and calculates power."""
    ber = np.zeros(len(snr_range_db))
    # Total power in mW
    power = P_RF*(num_tx + num_rx) + P_MMSE*num_subcarriers*(num_tx**3) 
    
    print(f"\nSimulating MMSE MIMO {num_tx}x{num_rx} for BER...")
    for snr_idx, snr_db in enumerate(snr_range_db):
        noise_var = 10**(-snr_db/10) # Normalized noise power relative to signal power=1
        total_bits = total_errors = 0
        
        for _ in range(num_syms):
            # Transmitter: Generate bits and modulate for each Tx antenna
            bits = np.random.randint(0, 2, num_tx*num_subcarriers*bits_per_symbol)
            tx_syms_freq_domain = np.zeros((num_tx, num_subcarriers), dtype=complex)
            for i in range(num_tx):
                start_idx = i*num_subcarriers*bits_per_symbol
                end_idx = (i+1)*num_subcarriers*bits_per_symbol
                tx_syms_freq_domain[i, :] = qpsk_modulate(bits[start_idx:end_idx]) # Average symbol power is 1
            
            # Channel: Apply correlated fading and add noise
            H = generate_correlated_channel(num_rx, num_tx, num_subcarriers, tx_corr, rx_corr)
            
            rx_syms_freq_domain = np.zeros((num_rx, num_subcarriers), dtype=complex)
            for k in range(num_subcarriers):
                noise = np.sqrt(noise_var/2)*(np.random.randn(num_rx,1) + 1j*np.random.randn(num_rx,1))
                rx_syms_freq_domain[:,k] = (H[:,:,k] @ tx_syms_freq_domain[:,k].reshape(-1,1) + noise).flatten()
            
            # MMSE Equalization
            est_syms_freq_domain = np.zeros((num_tx, num_subcarriers), dtype=complex)
            for k in range(num_subcarriers):
                Hk = H[:,:,k]
                # MMSE weight matrix: W = (H^H * H + N_0 * I)^-1 * H^H
                # N_0 is noise variance, here noise_var is per real dim, so total noise power is noise_var
                W = np.linalg.inv(Hk.conj().T @ Hk + noise_var*np.eye(num_tx)) @ Hk.conj().T
                est_syms_freq_domain[:,k] = (W @ rx_syms_freq_domain[:,k].reshape(-1,1)).flatten()
            
            # BER Calculation
            rx_bits = np.concatenate([qpsk_demodulate(est_syms_freq_domain[i]) for i in range(num_tx)])
            total_errors += np.sum(bits != rx_bits)
            total_bits += len(bits)
        
        ber[snr_idx] = total_errors / total_bits
        print(f"  SNR={snr_db}dB, BER={ber[snr_idx]:.6f}")
    
    return ber, power

# --- MD-OFDM Simulation ---
def simulate_md_ofdm(num_tx, num_syms):
    """Simulates MD-OFDM for BER and calculates power."""
    ber = np.zeros(len(snr_range_db))
    # Total power in mW (MD-OFDM assumes 1 Rx antenna)
    power = P_RF*(num_tx + 1) + P_SEL*num_subcarriers*num_tx 
    # Power scaling for symbols to ensure comparable total transmitted energy
    md_ofdm_power_scale = np.sqrt(num_tx/2) 

    print(f"\nSimulating MD-OFDM {num_tx}x1 for BER...")
    for snr_idx, snr_db in enumerate(snr_range_db):
        noise_var = 10**(-snr_db/10)
        total_bits = total_errors = 0
        
        for _ in range(num_syms):
            # Transmitter: Generate bits for the single user stream
            bits = np.random.randint(0, 2, num_subcarriers*bits_per_symbol)
            user_qpsk_symbols = qpsk_modulate(bits, scale=md_ofdm_power_scale)
            
            # Channel and Antenna Selection: 1 Rx antenna for MD-OFDM
            # H is 1 x num_tx x num_subcarriers
            H = generate_correlated_channel(1, num_tx, num_subcarriers, tx_corr, 0) # rx_corr=0 for single Rx antenna if not explicitly needed
            # Select best Tx antenna for each subcarrier
            tx_allocation_map = np.argmax(np.abs(H[0])**2, axis=0) # H[0] is the 1xnum_tx submatrix for the first Rx antenna
            
            # Transmission: Construct actual Tx symbols for each physical antenna
            tx_syms_freq_domain_per_antenna = np.zeros((num_tx, num_subcarriers), dtype=complex)
            for k in range(num_subcarriers):
                assigned_tx_antenna = tx_allocation_map[k]
                tx_syms_freq_domain_per_antenna[assigned_tx_antenna, k] = user_qpsk_symbols[k]
            
            # Receiver: Receive signal at single Rx antenna
            rx_syms_freq_domain = np.zeros(num_subcarriers, dtype=complex)
            for k in range(num_subcarriers):
                Hk_selected = H[0, tx_allocation_map[k], k] # Channel gain for selected Tx antenna on subcarrier k
                noise = np.sqrt(noise_var/2)*(np.random.randn() + 1j*np.random.randn())
                rx_syms_freq_domain[k] = Hk_selected * user_qpsk_symbols[k] + noise # Only the selected symbol is transmitted
            
            # Equalization: Simple scalar division for the selected link
            # Need to divide by the channel gain and the original scaling factor
            est_user_symbols = np.zeros(num_subcarriers, dtype=complex)
            for k in range(num_subcarriers):
                Hk_selected = H[0, tx_allocation_map[k], k]
                epsilon = 1e-9 # Avoid division by zero for very weak channels
                est_user_symbols[k] = rx_syms_freq_domain[k] / (Hk_selected + epsilon)
            
            rx_bits = qpsk_demodulate(est_user_symbols / md_ofdm_power_scale) # Undo scaling for demodulation
            
            # BER Calculation
            total_errors += np.sum(bits != rx_bits)
            total_bits += len(bits)
        
        ber[snr_idx] = total_errors / total_bits
        print(f"  SNR={snr_db}dB, BER={ber[snr_idx]:.6f}")
    
    return ber, power

# --- PAPR Simulation Function ---
def simulate_papr(system_type, num_tx, num_rx, num_papr_symbols):
    """Simulates time-domain signals and collects PAPR values for CCDF."""
    all_papr_values = []
    
    # Use the same power scaling factor for MD-OFDM symbols as in its BER simulation
    md_ofdm_power_scale = np.sqrt(num_tx/2) 

    print(f"\nSimulating PAPR for {system_type} system ({num_tx} Tx antennas)...")

    for _ in range(num_papr_symbols):
        if system_type == 'MMSE':
            # Generate frequency-domain symbols for each Tx antenna (average power 1 per symbol)
            bits = np.random.randint(0, 2, num_tx * num_subcarriers * bits_per_symbol)
            tx_syms_freq_domain = np.zeros((num_tx, num_subcarriers), dtype=complex)
            for i in range(num_tx):
                start_idx = i * num_subcarriers * bits_per_symbol
                end_idx = (i + 1) * num_subcarriers * bits_per_symbol
                tx_syms_freq_domain[i, :] = qpsk_modulate(bits[start_idx:end_idx], scale=1.0) 

            # Perform IFFT for each Tx antenna's signal
            for i in range(num_tx):
                # Normalize IFFT output to ensure average power is 1 for PAPR calculation
                time_domain_signal = np.fft.ifft(tx_syms_freq_domain[i, :]) * np.sqrt(num_subcarriers) 

                peak_power = np.max(np.abs(time_domain_signal)**2)
                avg_power = np.mean(np.abs(time_domain_signal)**2)
                
                # Handle potential zero average power if signal is all zeros, though unlikely for MMSE
                if avg_power > 0:
                    papr_db = 10 * np.log10(peak_power / avg_power)
                    all_papr_values.append(papr_db)

        elif system_type == 'MD':
            # Generate frequency-domain symbols for the single user stream
            bits = np.random.randint(0, 2, num_subcarriers * bits_per_symbol)
            user_qpsk_symbols = qpsk_modulate(bits, scale=md_ofdm_power_scale)

            # Mock antenna selection for PAPR (channel doesn't matter, just allocation pattern)
            # This generates a random selection map for each symbol for PAPR calculation.
            tx_allocation_map = np.random.randint(0, num_tx, num_subcarriers) 

            # Construct frequency-domain signals for each *physical* Tx antenna
            tx_syms_freq_domain_per_antenna = np.zeros((num_tx, num_subcarriers), dtype=complex)
            for k in range(num_subcarriers):
                assigned_tx_antenna = tx_allocation_map[k]
                tx_syms_freq_domain_per_antenna[assigned_tx_antenna, k] = user_qpsk_symbols[k]
            
            # Perform IFFT for each Tx antenna that transmits any data
            for i in range(num_tx):
                # An antenna might not transmit on any subcarrier in a given symbol for MD-OFDM
                # Only calculate PAPR if the antenna is actually transmitting something
                if np.any(tx_syms_freq_domain_per_antenna[i, :] != 0):
                    # Normalize IFFT output for PAPR calculation
                    time_domain_signal = np.fft.ifft(tx_syms_freq_domain_per_antenna[i, :]) * np.sqrt(num_subcarriers)

                    peak_power = np.max(np.abs(time_domain_signal)**2)
                    avg_power = np.mean(np.abs(time_domain_signal)**2)
                    
                    if avg_power > 0: # Ensure average power is not zero
                        papr_db = 10 * np.log10(peak_power / avg_power)
                        all_papr_values.append(papr_db)
    
    # Calculate CCDF
    papr_values = np.array(all_papr_values)
    papr_thresholds = np.arange(0, 15, 0.1) # PAPR thresholds from 0dB to 15dB
    ccdf_probabilities = np.zeros_like(papr_thresholds, dtype=float)

    if len(papr_values) > 0:
        for i, threshold in enumerate(papr_thresholds):
            ccdf_probabilities[i] = np.sum(papr_values > threshold) / len(papr_values)
    else:
        print(f"Warning: No PAPR values collected for {system_type} system. CCDF will be all zeros.")

    return papr_thresholds, ccdf_probabilities

# --- Energy Efficiency Calculation ---
def calculate_ee(ber, se, power_mW, bw=1e6):
    """Calculates Energy Efficiency in Mbps/Joule."""
    effective_se = se * (1 - ber)
    # Convert power from mW to Watts for Bits/Joule
    return (effective_se * bw) / (power_mW * 1e-3)  

# --- Main Simulation ---
if __name__ == "__main__":
    # Simulate BER for both systems
    ber_mmse, power_mmse = simulate_mmse_mimo(4, 4, num_ofdm_symbols_ber)
    ber_md, power_md = simulate_md_ofdm(4, num_ofdm_symbols_ber)
    
    # Ideal Spectral Efficiency (bits per OFDM symbol per Hz)
    # For a bandwidth of 1 Hz, these are bits per symbol.
    se_mmse = 4 * bits_per_symbol  # 4 spatial streams, 2 bits/symbol/stream
    se_md = 1 * bits_per_symbol         # 1 spatial stream, 2 bits/symbol/stream
    
    # Calculate Energy Efficiency
    # We pass the ideal spectral efficiency per unit bandwidth (e.g., 1 Hz).
    # The bandwidth (bw=1e6) in calculate_ee scales it to Mbps/Joule.
    ee_mmse = [calculate_ee(ber_val, se_mmse, power_mmse) for ber_val in ber_mmse]
    ee_md = [calculate_ee(ber_val, se_md, power_md) for ber_val in ber_md] 
    
    # Simulate PAPR for both systems
    papr_thresholds_mmse, ccdf_mmse = simulate_papr('MMSE', 4, 4, num_ofdm_symbols_papr)
    papr_thresholds_md, ccdf_md = simulate_papr('MD', 4, 1, num_ofdm_symbols_papr) 

    # --- Plot Results (Separate Figures with Increased Font Size) ---
    
    # BER Plot
    plt.figure(figsize=(8, 6)) # Adjust figure size for a single plot
    plt.semilogy(snr_range_db, ber_mmse, 'b-o', label=f'MMSE 4x4 (Power={power_mmse:.1f}mW)')
    plt.semilogy(snr_range_db, ber_md, 'r--s', label=f'MD-OFDM 4x1 (Power={power_md:.1f}mW)')
    plt.xlabel('SNR (dB)', fontsize=plot_font_size)
    plt.ylabel('BER', fontsize=plot_font_size)
    plt.grid(True)
    plt.legend(fontsize=legend_font_size)
    # plt.title('BER Comparison', fontsize=title_font_size)
    plt.ylim(1e-5, 1) 
    plt.tick_params(axis='both', which='major', labelsize=plot_font_size) # Adjust tick label size
    plt.tight_layout()
    plt.savefig('ber_vs_snr_plot.png') # Save as PDF for LaTeX
    plt.show()
    
    # Energy Efficiency Plot
    plt.figure(figsize=(8, 6)) 
    plt.plot(snr_range_db, ee_mmse, 'b-o', label='MMSE 4x4')
    plt.plot(snr_range_db, ee_md, 'r--s', label='MD-OFDM 4x1')
    plt.xlabel('SNR (dB)', fontsize=plot_font_size)
    plt.ylabel('Energy Efficiency (Mbps/Joule)', fontsize=plot_font_size)
    plt.grid(True)
    plt.legend(fontsize=legend_font_size)
    # plt.title('Energy Efficiency Comparison', fontsize=title_font_size)
    min_ee = min(min(ee_mmse), min(ee_md))
    max_ee = max(max(ee_mmse), max(ee_md))
    plt.ylim(min_ee * 0.9, max_ee * 1.1)
    plt.tick_params(axis='both', which='major', labelsize=plot_font_size)
    plt.tight_layout()
    plt.savefig('ee_vs_snr_plot.png') # Save as PDF for LaTeX
    plt.show()
    
    # PAPR CCDF Plot
    plt.figure(figsize=(8, 6))
    plt.semilogy(papr_thresholds_mmse, ccdf_mmse, 'b-o', label='MMSE 4x4')
    plt.semilogy(papr_thresholds_md, ccdf_md, 'r--s', label='MD-OFDM 4x1')
    plt.xlabel('PAPR (dB)', fontsize=plot_font_size)
    plt.ylabel('CCDF (Prob(PAPR > threshold))', fontsize=plot_font_size)
    plt.grid(True)
    plt.legend(fontsize=legend_font_size)
    # plt.title('PAPR CCDF Comparison', fontsize=title_font_size)
    plt.ylim(1e-4, 1) 
    plt.xlim(0, 12) 
    plt.tick_params(axis='both', which='major', labelsize=plot_font_size)
    plt.tight_layout()
    plt.savefig('ccdf_vs_papr_plot.png') # Save as PDF for LaTeX
    plt.show()

    print("\nSimulation complete. BER, Energy Efficiency, and PAPR CCDF plots displayed.")

    # --- Print Energy Efficiency Values (for confirmation/debugging) ---
    print("\n--- Energy Efficiency (Mbps/Joule) ---")
    print(f"{'SNR (dB)':<10} | {'MMSE 4x4 EE':<15} | {'MD-OFDM 4x1 EE':<15}")
    print("-" * 45)
    for i in range(len(snr_range_db)):
        print(f"{snr_range_db[i]:<10.1f} | {ee_mmse[i]:<15.4f} | {ee_md[i]:<15.4f}")

    # --- Print PAPR CCDF Values (for confirmation/debugging) ---
    print("\n--- PAPR CCDF (Prob(PAPR > threshold)) ---")
    step_for_printing = 10 
    print(f"{'PAPR (dB)':<10} | {'MMSE 4x4 CCDF':<18} | {'MD-OFDM 4x1 CCDF':<18}")
    print("-" * 50)
    for i in range(0, len(papr_thresholds_mmse), step_for_printing):
        print(f"{papr_thresholds_mmse[i]:<10.1f} | {ccdf_mmse[i]:<18.6e} | {ccdf_md[i]:<18.6e}")

# END